# 05c · Stock, objectifs et ventes en ligne détaillées : diagnostic et variables

**Projet 4 : Analyse des données de vente et retours** · Sprint 3

**Objectif** : appliquer aux trois tables de faits ajoutées avec l'année de référence la même démarche qu'aux ventes et aux charges : structure, qualité, rapprochement avec nos ventes, nature des données, décisions, variables. Pour chacune, on répond à une question simple : **que peut-on lui faire dire, et que ne peut-on pas lui faire dire ?**

| Partie | Table | Une ligne = | Question posée |
|---|---|---|---|
| A | Stock (`FactInventory`) | un relevé du dimanche, pour un produit dans une boutique | Le stock dit-il quels produits sont présents dans un magasin ? |
| B | Objectifs de vente (`FactSalesQuota`) | une boutique, un produit, un mois, un scénario | L'objectif est-il un vrai objectif, fixé avant les ventes ? |
| C | Ventes en ligne détaillées (`FactOnlineSales`) | une unité commandée par un client | Ce détail est-il celui des ventes de la boutique en ligne ? |

**Pourquoi ce notebook.** L'analyse cherche à expliquer la baisse des ventes des magasins entre 2019 et 2021 (`05b_charges`, section 7). Une des pistes est que l'offre en magasin s'est réduite. Avant de s'appuyer sur la table de stock pour le montrer, il faut savoir ce qu'elle mesure. Même prudence pour les objectifs et pour le détail des ventes en ligne.

**Lit dans** : `data/interim` (`stock`, `objectifs`, `ventes_en_ligne`, produits par `01_etl`), `data/clean` (ventes avec variables, dimensions, charges)

**Écrit dans** : `data/clean` (`fait_stock`, `fait_objectifs`, `fait_ventes_en_ligne`), `data/clean/agregats` (`assortiment_boutique_mois`, `produit_annee`), `reports/livrables/L3_diagnostic_nettoyage/diagnostic_autres_tables.xlsx`, `reports/figures`

**Ce notebook ne modifie aucune valeur.** Il observe, il décide de l'usage de chaque table, et il prépare deux agrégats pour l'analyse.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins, année de référence, exercice étudié
#   viz    -> charte graphique
from src import config, viz

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 70)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Racine du projet :", config.ROOT)

EXERCICE, REFERENCE = config.EXERCICE, config.ANNEE_REFERENCE
LIBELLES_TYPES = {"Store": "Magasins", "Online": "En ligne", "Reseller": "Revendeur"}
TOLERANCE = 0.02      # les montants sont arrondis au cent dans la source

# ---------------------------------------------------------------------------
# 3. Registre des contrôles (même principe que dans 03_diagnostic et 05b_charges)
# ---------------------------------------------------------------------------
registre = []


def noter(table, controle, nb, gravite, action, total=None, zero_est_ok=True):
    """Ajoute un contrôle au registre et l'affiche.

    table    : table concernée (Stock, Objectifs, Ventes en ligne)
    controle : ce qui est vérifié
    nb       : nombre de cas concernés
    gravite  : "OK" si nb vaut 0 (sauf zero_est_ok=False), sinon "Information", "À décider" ou "À corriger"
    action   : ce qu'on propose de faire
    total    : nombre total de cas, pour afficher une part
    """
    nb = int(nb)
    if nb == 0 and zero_est_ok:
        gravite = "OK"
    part = f" ({nb / total:.2%})" if total else ""
    registre.append({"table": table, "controle": controle, "nb": nb,
                     "part": nb / total if total else np.nan, "gravite": gravite, "action": action})
    nombre = f"{nb:,}".replace(",", " ")
    print(f"[{gravite:<11}] {table} · {controle} : {nombre}{part}")


def espaces(nombre, decimales=0):
    """Nombre avec une espace comme séparateur de milliers."""
    return f"{nombre:,.{decimales}f}".replace(",", " ")

## 1. Chargement

Les trois tables ont été lues et mises en forme par `01_etl` (section 8) : noms de colonnes en français, dates décalées de 12 ans, types corrects. On charge aussi les ventes propres (pour les rapprochements) et les dimensions.

In [ ]:
chemins = {nom: config.INTERIM / f"{nom}.parquet" for nom in ["stock", "objectifs", "ventes_en_ligne"]}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : lancer python -m src.extraction, puis 01_etl.")

stock = pd.read_parquet(chemins["stock"])
objectifs = pd.read_parquet(chemins["objectifs"])
en_ligne = pd.read_parquet(chemins["ventes_en_ligne"])

ventes = pd.read_parquet(config.CLEAN / "fait_ventes_variables.parquet")
dim_boutique = pd.read_parquet(config.CLEAN_DIMENSIONS / "dim_boutique.parquet")
dim_produit = pd.read_parquet(config.CLEAN_DIMENSIONS / "dim_produit.parquet")
type_boutique = dim_boutique.set_index("id_boutique").type_boutique

for table in (stock, objectifs, en_ligne):
    table["annee"] = table.date.dt.year
ventes["mois_debut"] = ventes.date.dt.to_period("M").dt.to_timestamp()      # premier jour du mois de la vente

inventaire = pd.DataFrame({
    "lignes": [len(stock), len(objectifs), len(en_ligne), len(ventes)],
    "colonnes": [stock.shape[1], objectifs.shape[1], en_ligne.shape[1], ventes.shape[1]],
    "du": [t.date.min().date() for t in (stock, objectifs, en_ligne, ventes)],
    "au": [t.date.max().date() for t in (stock, objectifs, en_ligne, ventes)],
    "boutiques": [t.id_boutique.nunique() for t in (stock, objectifs, en_ligne, ventes)],
    "produits": [t.id_produit.nunique() for t in (stock, objectifs, en_ligne, ventes)],
    "cellules vides": [int(t.isna().sum().sum()) for t in (stock, objectifs, en_ligne)] + [np.nan],
}, index=["Stock", "Objectifs", "Ventes en ligne détaillées", "Ventes (rappel)"])
display(inventaire.style.format({"lignes": "{:,.0f}", "cellules vides": "{:,.0f}"}, na_rep=""))
for nom, table in [("Stock", stock), ("Objectifs", objectifs), ("Ventes en ligne", en_ligne)]:
    noter(nom, "Cellules vides", table.isna().sum().sum(), "À corriger", "Vérifier l'extraction", total=table.size)

# Partie A · Le stock

## 2. Structure et qualité

La table contient, pour un produit dans une boutique à une date donnée : la quantité en stock, la quantité en commande chez le fournisseur, le stock de sécurité, le coût unitaire, et quatre indicateurs d'ancienneté du stock.

In [ ]:
display(stock.head(3))
display(stock[["quantite_en_stock", "quantite_en_commande", "stock_de_securite", "cout_unitaire",
               "jours_en_stock", "jours_en_stock_min", "jours_en_stock_max", "anciennete"]].describe().T)

# ---------------------------------------------------------------------------
# 1. Clé, dates, valeurs impossibles
# ---------------------------------------------------------------------------
CLE_STOCK = ["date", "id_boutique", "id_produit"]
noter("Stock", "Lignes en double sur la clé (date, boutique, produit)", stock.duplicated(CLE_STOCK).sum(),
      "À corriger", "Dédoublonner", total=len(stock))
dates_releves = pd.Series(sorted(stock.date.unique()))
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
print("Jour de la semaine des relevés :", {JOURS[j]: int(n) for j, n in dates_releves.dt.dayofweek.value_counts().items()},
      "· écart entre deux relevés :", {f"{int(j)} jours": int(n) for j, n in dates_releves.diff().dt.days.value_counts().items()})
noter("Stock", "Semaines sans aucun relevé sur la période", int((dates_releves.diff().dt.days > 7).sum()),
      "À décider", "Vérifier l'extraction", total=len(dates_releves))
for col in ["quantite_en_stock", "quantite_en_commande", "stock_de_securite"]:
    noter("Stock", f"{col} négatif", (stock[col] < 0).sum(), "À décider", "Vérifier", total=len(stock))
noter("Stock", "Relevés avec une quantité en stock nulle (rupture constatée)", (stock.quantite_en_stock == 0).sum(),
      "Information", "Aucune : les ruptures enregistrées sont presque inexistantes", total=len(stock), zero_est_ok=False)

# ---------------------------------------------------------------------------
# 2. Jointures : toutes les boutiques et tous les produits du stock sont-ils connus ?
# ---------------------------------------------------------------------------
noter("Stock", "Boutiques du stock absentes de nos ventes", len(set(stock.id_boutique) - set(ventes.id_boutique)),
      "À corriger", "Vérifier le périmètre de l'extraction")
noter("Stock", "Produits du stock absents de la dimension produit", len(set(stock.id_produit) - set(dim_produit.id_produit)),
      "À corriger", "Vérifier la dimension")
cout_catalogue = stock.id_produit.map(dim_produit.set_index("id_produit").cout_catalogue)
noter("Stock", "Relevés dont le coût unitaire diffère du coût catalogue", ((stock.cout_unitaire - cout_catalogue).abs() > TOLERANCE).sum(),
      "À décider", "Choisir le coût de référence", total=len(stock))

# ---------------------------------------------------------------------------
# 3. Par année : relevés, boutiques, produits, niveaux moyens
# ---------------------------------------------------------------------------
par_annee = stock.groupby("annee").agg(
    lignes=("date", "size"), dates_de_releve=("date", "nunique"), boutiques=("id_boutique", "nunique"),
    produits=("id_produit", "nunique"), quantite_moyenne_en_stock=("quantite_en_stock", "mean"),
    stock_de_securite_moyen=("stock_de_securite", "mean"), quantite_moyenne_en_commande=("quantite_en_commande", "mean"))
par_annee["part sous le stock de sécurité"] = (stock.quantite_en_stock < stock.stock_de_securite).groupby(stock.annee).mean()
display(par_annee.style.format({"lignes": "{:,.0f}", "quantite_moyenne_en_stock": "{:.1f}", "stock_de_securite_moyen": "{:.1f}",
                                "quantite_moyenne_en_commande": "{:.1f}", "part sous le stock de sécurité": "{:.1%}"}))

# Boutiques sans aucun relevé une année donnée : est-ce cohérent avec leur date d'ouverture ?
presence = stock.groupby(["id_boutique", "annee"]).size().unstack("annee")
absentes = presence[presence.isna().any(axis=1)].join(dim_boutique.set_index("id_boutique")[["nom_boutique", "date_ouverture"]])
premier_releve = stock.groupby("id_boutique").date.min().rename("premier_releve")
print("Boutiques sans relevé de stock une année au moins :")
display(absentes.join(premier_releve)[["nom_boutique", "date_ouverture", "premier_releve"] + sorted(stock.annee.unique())]
        .style.format({a: "{:,.0f}" for a in stock.annee.unique()} | {"date_ouverture": "{:%Y-%m-%d}", "premier_releve": "{:%Y-%m-%d}"},
                      na_rep="aucun relevé"))
releve_avant_ouverture = (stock.date < stock.id_boutique.map(dim_boutique.set_index("id_boutique").date_ouverture)).sum()
noter("Stock", "Relevés datés avant l'ouverture de la boutique", releve_avant_ouverture, "Information",
      "Aucune", total=len(stock))

**Lecture**

- La table est **propre** : aucun vide, aucun doublon, aucune quantité négative, un relevé chaque dimanche des trois années, et le coût unitaire est celui du catalogue.
- Elle est **cohérente avec les dates d'ouverture** : les magasins ouverts en 2020 et 2021 n'ont aucun relevé avant leur ouverture. Contrairement aux ventes et au plan, la table de stock ne contient aucune ligne d'avant ouverture. C'est un argument de plus pour considérer les 189 ventes d'avant ouverture comme une anomalie (règle N8).
- **Les ruptures enregistrées sont presque inexistantes** : 210 relevés à zéro sur 1,5 million. La table ne permet donc pas d'étudier les ruptures de stock au sens habituel (un produit référencé mais absent du rayon).
- **Le nombre de lignes est divisé par trois en deux ans** (775 133, 488 815, 253 740), alors que le nombre de boutiques et de produits augmente. Avant d'y lire une réduction du stock, il faut comprendre ce qu'est une ligne : c'est l'objet de la section suivante.

## 3. Ce que la table contient vraiment : un relevé partiel

Si la table était un inventaire complet, chaque produit présent dans une boutique aurait un relevé chaque dimanche, soit 52 par an. On compte donc le nombre de relevés par produit, boutique et année, puis on confronte la table aux ventes : **un produit vendu dans une boutique un mois donné a-t-il un relevé de stock ce mois-là ?** Par construction, un produit vendu était présent.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Nombre de relevés par (boutique, produit, année) : 52 si l'inventaire était complet
# ---------------------------------------------------------------------------
releves = stock.groupby(["annee", "id_boutique", "id_produit"]).size().rename("releves")
densite = releves.groupby("annee").describe(percentiles=[0.25, 0.5, 0.75])[["count", "mean", "25%", "50%", "75%", "max"]]
densite["part des couples avec les 52 relevés"] = (releves == 52).groupby("annee").mean()
print("Nombre de relevés dans l'année, par couple (boutique, produit) présent dans la table :")
display(densite.rename(columns={"count": "couples boutique-produit", "mean": "moyenne", "50%": "médiane", "max": "maximum"})
        .style.format({"couples boutique-produit": "{:,.0f}", "moyenne": "{:.1f}", "25%": "{:.0f}", "médiane": "{:.0f}",
                       "75%": "{:.0f}", "maximum": "{:.0f}", "part des couples avec les 52 relevés": "{:.1%}"}))

# Exemple : les relevés d'un produit dans une boutique, la première année
exemple = stock[(stock.id_boutique == stock.id_boutique.iloc[0]) & (stock.id_produit == stock.id_produit.iloc[0])]
print("Exemple, dates des relevés d'un même produit dans une même boutique :",
      [f"{d:%Y-%m-%d}" for d in exemple.date.sort_values().head(12)])

# ---------------------------------------------------------------------------
# 2. Confrontation avec les ventes, au niveau boutique × produit × mois
# ---------------------------------------------------------------------------
stock["mois_debut"] = stock.date.dt.to_period("M").dt.to_timestamp()
vendus = (ventes.groupby(["id_boutique", "id_produit", "mois_debut"]).quantite_vente.sum().rename("unites_vendues").reset_index())
releves_mois = (stock.groupby(["id_boutique", "id_produit", "mois_debut"])
                .agg(releves=("date", "size"), stock_moyen=("quantite_en_stock", "mean")).reset_index())
croise = vendus.merge(releves_mois, on=["id_boutique", "id_produit", "mois_debut"], how="outer", indicator=True)
croise["annee"] = croise.mois_debut.dt.year
croise["type"] = croise.id_boutique.map(type_boutique).map(LIBELLES_TYPES)
croise["situation"] = croise["_merge"].map({"both": "vendu et relevé", "left_only": "vendu sans relevé", "right_only": "relevé sans vente"})

lignes = []
for (type_, annee), g in croise.groupby(["type", "annee"]):
    vendu = g[g.situation != "relevé sans vente"]
    lignes.append({"type": type_, "annee": annee,
                   "produits-mois vendus": len(vendu),
                   "dont avec un relevé de stock le même mois": (vendu.situation == "vendu et relevé").mean(),
                   "unités vendues sans relevé le même mois": vendu.loc[vendu.situation == "vendu sans relevé", "unites_vendues"].sum() / vendu.unites_vendues.sum(),
                   "produits-mois relevés": int((g.situation != "vendu sans relevé").sum()),
                   "dont sans aucune vente le même mois": (g.situation == "relevé sans vente").sum() / (g.situation != "vendu sans relevé").sum()})
couverture_mois = pd.DataFrame(lignes).set_index(["type", "annee"]).loc[["Magasins", "En ligne", "Revendeur"]]
print("Un produit vendu dans une boutique un mois donné a-t-il un relevé de stock ce mois-là ?")
display(couverture_mois.style.format({"produits-mois vendus": "{:,.0f}", "dont avec un relevé de stock le même mois": "{:.1%}",
                                      "unités vendues sans relevé le même mois": "{:.1%}", "produits-mois relevés": "{:,.0f}",
                                      "dont sans aucune vente le même mois": "{:.1%}"}))

# ---------------------------------------------------------------------------
# 3. Même confrontation sur l'année entière : le produit a-t-il au moins UN relevé dans l'année ?
# ---------------------------------------------------------------------------
vendus_an = ventes.groupby(["id_boutique", "id_produit", "annee"]).quantite_vente.sum().rename("unites_vendues").reset_index()
releves_an = releves.reset_index()
croise_an = vendus_an.merge(releves_an, on=["id_boutique", "id_produit", "annee"], how="outer", indicator=True)
croise_an["type"] = croise_an.id_boutique.map(type_boutique).map(LIBELLES_TYPES)
lignes = []
for (type_, annee), g in croise_an.groupby(["type", "annee"]):
    vendu = g[g["_merge"] != "right_only"]
    lignes.append({"type": type_, "annee": annee,
                   "couples boutique-produit vendus": len(vendu),
                   "dont sans aucun relevé dans l'année": (vendu["_merge"] == "left_only").mean(),
                   "unités vendues sans aucun relevé dans l'année": vendu.loc[vendu["_merge"] == "left_only", "unites_vendues"].sum() / vendu.unites_vendues.sum(),
                   "couples relevés sans aucune vente dans l'année": int((g["_merge"] == "right_only").sum())})
couverture_an = pd.DataFrame(lignes).set_index(["type", "annee"]).loc[["Magasins", "En ligne", "Revendeur"]]
print("Un produit vendu dans une boutique a-t-il au moins un relevé de stock dans l'année ?")
display(couverture_an.style.format({"couples boutique-produit vendus": "{:,.0f}", "dont sans aucun relevé dans l'année": "{:.1%}",
                                    "unités vendues sans aucun relevé dans l'année": "{:.1%}",
                                    "couples relevés sans aucune vente dans l'année": "{:,.0f}"}))

sans_releve = croise_an[(croise_an["_merge"] == "left_only") & (croise_an.annee == EXERCICE) & (croise_an.type == "Magasins")]
noter("Stock", f"Couples (magasin, produit) vendus en {EXERCICE} sans aucun relevé de stock dans l'année", len(sans_releve),
      "Information", "Ne pas déduire l'absence d'un produit de l'absence de relevé",
      total=int(((croise_an["_merge"] != "right_only") & (croise_an.annee == EXERCICE) & (croise_an.type == "Magasins")).sum()))

**Lecture**

- **La table n'est pas un inventaire complet.** Un produit présent dans une boutique n'a pas 52 relevés par an, mais une dizaine en 2019, six ou sept en 2020 et cinq en 2021 en moyenne, groupés par séries de quelques dimanches consécutifs. Aucun couple n'a ses 52 relevés. Les autres semaines, le produit n'a pas de ligne : cela ne veut pas dire qu'il n'est pas en magasin.
- **La preuve par les ventes.** En 2019, 92 % des produits vendus dans un magasin un mois donné ont un relevé de stock ce mois-là. En 2021, ce n'est plus que 62 %. Sur l'année entière, 27 % des couples (magasin, produit) vendus en 2021 n'ont **aucun** relevé de stock de l'année, et ils représentent 23 % des unités vendues par les magasins. Ces produits étaient bien en magasin, puisqu'ils ont été vendus.
- **La table s'éclaircit d'année en année**, et plus vite que les ventes. La division par trois du nombre de lignes mesure donc d'abord la **densité des relevés**, pas la quantité de produits en rayon.
- **À l'inverse**, près de la moitié des produits relevés dans un magasin un mois donné n'ont aucune vente ce mois-là : un produit peut être en stock sans se vendre. C'est l'information utile de cette table.

**Ce qu'on peut faire dire à la table de stock**

| Usage | Possible ? | Pourquoi |
|---|---|---|
| Dire qu'un produit **était présent** dans une boutique à une date | Oui | Un relevé existe, avec une quantité positive |
| Dire qu'un produit **était absent** d'une boutique | **Non** | L'absence de relevé est la règle, même pour un produit vendu |
| Compter les produits en stock dans une boutique une semaine donnée | **Non** | Le compte suit la densité des relevés, qui change d'une année à l'autre |
| Mesurer le niveau de stock d'un produit quand il est relevé (quantité, stock de sécurité, commande en cours) | Oui | Les quantités sont cohérentes et sans valeur impossible |
| Repérer les produits en stock qui ne se vendent pas | Oui, au niveau de l'année | Un relevé sans vente du même couple dans l'année |
| Étudier les ruptures | **Non** | 210 relevés à zéro sur 1,5 million |

**Conséquence pour l'analyse de l'offre.** La thèse « les magasins vendent moins parce qu'ils proposent moins de produits » ne peut pas être démontrée par un comptage des produits en stock : ce comptage baisserait même si l'offre n'avait pas changé. La mesure de l'offre doit venir des **ventes elles-mêmes** (produits vendus par magasin et par mois, par an) et du **catalogue** (dates de mise en vente). Le stock sert de complément, pour les produits qu'il relève. C'est ce que prépare la section suivante.

## 4. Variables : l'offre par boutique et par mois, et la vie de chaque produit

Deux agrégats sont préparés pour l'analyse.

**`assortiment_boutique_mois`** : une ligne par boutique et par mois.

| Variable | Définition | Source |
|---|---|---|
| `produits_vendus` | nombre de produits différents vendus dans le mois | ventes |
| `lignes_de_vente`, `unites_vendues`, `ca_brut` | activité du mois | ventes |
| `produits_releves` | nombre de produits différents ayant au moins un relevé de stock dans le mois | stock |
| `releves` | nombre de relevés du mois | stock |
| `stock_moyen_par_releve` | quantité moyenne en stock par relevé | stock |
| `valeur_stock_moyenne_par_releve` | quantité × coût unitaire, en moyenne par relevé | stock |
| `part_releves_sous_securite` | part des relevés sous le stock de sécurité | stock |
| `produits_vendus_et_releves` | produits à la fois vendus et relevés dans le mois | les deux |

**`produit_annee`** : une ligne par produit, type de boutique et année.

| Variable | Définition |
|---|---|
| `boutiques_qui_vendent` | nombre de boutiques où le produit a au moins une vente dans l'année |
| `mois_avec_vente`, `lignes_de_vente`, `unites_vendues`, `ca_brut` | activité du produit |
| `boutiques_avec_releve` | nombre de boutiques où le produit a au moins un relevé de stock dans l'année |
| `date_mise_en_vente`, `annee_de_lancement` | date du catalogue |
| `premiere_vente`, `derniere_vente` | sur l'ensemble de la période et de nos boutiques |

In [ ]:
# ---------------------------------------------------------------------------
# 1. Assortiment par boutique et par mois
# ---------------------------------------------------------------------------
cles = ["id_boutique", "mois_debut"]
cote_ventes = ventes.groupby(cles).agg(produits_vendus=("id_produit", "nunique"), lignes_de_vente=("id_vente", "size"),
                                       unites_vendues=("quantite_vente", "sum"), ca_brut=("ca_brut", "sum"))
stock["valeur_stock"] = stock.quantite_en_stock * stock.cout_unitaire
stock["sous_securite"] = stock.quantite_en_stock < stock.stock_de_securite
cote_stock = stock.groupby(cles).agg(produits_releves=("id_produit", "nunique"), releves=("date", "size"),
                                     stock_moyen_par_releve=("quantite_en_stock", "mean"),
                                     valeur_stock_moyenne_par_releve=("valeur_stock", "mean"),
                                     part_releves_sous_securite=("sous_securite", "mean"))
les_deux = (croise[croise.situation == "vendu et relevé"].groupby(cles).size().rename("produits_vendus_et_releves"))
assortiment = cote_ventes.join(cote_stock, how="outer").join(les_deux, how="left").reset_index()
for col in ["produits_vendus", "lignes_de_vente", "unites_vendues", "ca_brut", "produits_releves", "releves", "produits_vendus_et_releves"]:
    assortiment[col] = assortiment[col].fillna(0)
assortiment = assortiment.merge(dim_boutique[["id_boutique", "nom_boutique", "type_boutique", "pays", "date_ouverture"]], on="id_boutique", how="left")
assortiment["annee"] = assortiment.mois_debut.dt.year
assortiment["mois"] = assortiment.mois_debut.dt.month
assortiment["mois_avant_ouverture"] = assortiment.mois_debut < assortiment.date_ouverture.dt.to_period("M").dt.to_timestamp()
assortiment["annee_complete"] = assortiment.date_ouverture < pd.to_datetime(assortiment.annee.astype(str) + "-01-02")
assortiment = assortiment.drop(columns="date_ouverture")
print(f"assortiment_boutique_mois : {espaces(len(assortiment))} lignes × {assortiment.shape[1]} colonnes")

# Lecture rapide : magasins ouverts toute l'année, moyenne par magasin et par mois
magasins = assortiment[(assortiment.type_boutique == "Store") & assortiment.annee_complete]
resume = magasins.groupby("annee").agg(
    magasins=("id_boutique", "nunique"),
    produits_vendus_par_mois=("produits_vendus", "mean"), lignes_de_vente_par_mois=("lignes_de_vente", "mean"),
    unites_vendues_par_mois=("unites_vendues", "mean"), ca_brut_par_mois=("ca_brut", "mean"),
    produits_releves_par_mois=("produits_releves", "mean"), stock_moyen_par_releve=("stock_moyen_par_releve", "mean"))
resume["unités par ligne de vente"] = resume.unites_vendues_par_mois / resume.lignes_de_vente_par_mois
resume["lignes par produit vendu"] = resume.lignes_de_vente_par_mois / resume.produits_vendus_par_mois
print("Magasins ouverts toute l'année : moyenne par magasin et par mois")
display(resume.style.format({"produits_vendus_par_mois": "{:.0f}", "lignes_de_vente_par_mois": "{:.0f}", "unites_vendues_par_mois": "{:,.0f}",
                             "ca_brut_par_mois": "{:,.0f}", "produits_releves_par_mois": "{:.0f}", "stock_moyen_par_releve": "{:.1f}",
                             "unités par ligne de vente": "{:.1f}", "lignes par produit vendu": "{:.2f}"}))

# Même lecture sur l'année entière : combien de produits différents un magasin vend-il dans l'année ?
par_an = (ventes[ventes.type_boutique == "Store"].groupby(["annee", "id_boutique"])
          .agg(produits_vendus_dans_l_annee=("id_produit", "nunique"), lignes=("id_vente", "size")).reset_index())
par_an = par_an.merge(assortiment[["id_boutique", "annee", "annee_complete"]].drop_duplicates(), on=["id_boutique", "annee"])
print("Magasins ouverts toute l'année : produits différents vendus dans l'année, par magasin")
display(par_an[par_an.annee_complete].groupby("annee").produits_vendus_dans_l_annee.describe()[["count", "mean", "min", "50%", "max"]]
        .style.format("{:,.0f}"))
catalogue_vendu = ventes.groupby(["annee", "type_boutique"]).id_produit.nunique().unstack("type_boutique").rename(columns=LIBELLES_TYPES)
catalogue_vendu["Toutes boutiques"] = ventes.groupby("annee").id_produit.nunique()
print("Produits différents vendus dans l'année, par type de boutique (tous magasins confondus) :")
display(catalogue_vendu)

In [ ]:
# ---------------------------------------------------------------------------
# 2. Vie de chaque produit, par type de boutique et par année
# ---------------------------------------------------------------------------
ventes["type"] = ventes.type_boutique.map(LIBELLES_TYPES)
stock["type"] = stock.id_boutique.map(type_boutique).map(LIBELLES_TYPES)
cle_produit = ["id_produit", "type", "annee"]
produit_annee = ventes.groupby(cle_produit).agg(
    boutiques_qui_vendent=("id_boutique", "nunique"), mois_avec_vente=("mois_debut", "nunique"),
    lignes_de_vente=("id_vente", "size"), unites_vendues=("quantite_vente", "sum"), ca_brut=("ca_brut", "sum"),
    marge_nette=("marge_nette", "sum"))
produit_annee = produit_annee.join(stock.groupby(cle_produit).agg(boutiques_avec_releve=("id_boutique", "nunique"),
                                                                    releves=("date", "size")), how="outer").reset_index()
for col in ["boutiques_qui_vendent", "mois_avec_vente", "lignes_de_vente", "unites_vendues", "ca_brut", "marge_nette",
            "boutiques_avec_releve", "releves"]:
    produit_annee[col] = produit_annee[col].fillna(0)
vie = ventes.groupby("id_produit").date.agg(premiere_vente="min", derniere_vente="max")
produit_annee = (produit_annee
                 .merge(dim_produit[["id_produit", "nom_produit", "categorie", "sous_categorie", "marque", "date_mise_en_vente"]],
                        on="id_produit", how="left")
                 .merge(vie, on="id_produit", how="left"))
produit_annee["annee_de_lancement"] = produit_annee.date_mise_en_vente.dt.year
print(f"produit_annee : {espaces(len(produit_annee))} lignes × {produit_annee.shape[1]} colonnes")

# Le catalogue : produits mis en vente chaque année, et date de leur première vente dans nos boutiques
lancements = dim_produit[dim_produit.id_produit.isin(ventes.id_produit.unique())].copy()
lancements["annee_de_lancement"] = lancements.date_mise_en_vente.dt.year
lancements = lancements.merge(vie, on="id_produit")
lancements["jours_entre_mise_en_vente_et_premiere_vente"] = (lancements.premiere_vente - lancements.date_mise_en_vente).dt.days
print("Produits du catalogue vendus par nos boutiques, selon leur année de mise en vente :")
display(lancements.groupby("annee_de_lancement").agg(
    produits=("id_produit", "size"), derniere_mise_en_vente=("date_mise_en_vente", "max"),
    delai_median_avant_premiere_vente_jours=("jours_entre_mise_en_vente_et_premiere_vente", "median"),
    premiere_vente_avant_mise_en_vente=("jours_entre_mise_en_vente_et_premiere_vente", lambda s: int((s < 0).sum())))
    .style.format({"derniere_mise_en_vente": "{:%Y-%m-%d}", "delai_median_avant_premiere_vente_jours": "{:.0f}"}))
noter("Stock", "Produits vendus avant leur date de mise en vente au catalogue",
      (lancements.jours_entre_mise_en_vente_et_premiere_vente < 0).sum(), "Information",
      "À connaître avant d'utiliser la date de mise en vente", total=len(lancements))

# Dans combien de magasins un produit est-il vendu, selon l'année ?
diffusion = (produit_annee[(produit_annee.type == "Magasins") & (produit_annee.boutiques_qui_vendent > 0)]
             .groupby("annee").agg(produits_vendus_en_magasin=("id_produit", "nunique"),
                                   magasins_par_produit_moyenne=("boutiques_qui_vendent", "mean"),
                                   magasins_par_produit_mediane=("boutiques_qui_vendent", "median"),
                                   mois_avec_vente_par_produit=("mois_avec_vente", "mean"),
                                   lignes_par_produit=("lignes_de_vente", "mean")))
print("Diffusion d'un produit dans les magasins, par année :")
display(diffusion.style.format({"magasins_par_produit_moyenne": "{:.1f}", "magasins_par_produit_mediane": "{:.0f}",
                                "mois_avec_vente_par_produit": "{:.1f}", "lignes_par_produit": "{:.0f}"}))

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : deux mesures de l'offre d'un magasin, qui ne racontent pas la même chose
# ---------------------------------------------------------------------------
annees = sorted(resume.index)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
series = [(resume.produits_releves_par_mois, "Produits avec un relevé de stock\ndans le mois", viz.GRIS),
          (resume.produits_vendus_par_mois, "Produits différents vendus\ndans le mois", viz.PRINCIPALE),
          (par_an[par_an.annee_complete].groupby("annee").produits_vendus_dans_l_annee.mean(), "Produits différents vendus\ndans l'année", viz.ACCENT)]
for ax, (serie, libelle, couleur) in zip(axes, series):
    ax.bar([str(a) for a in annees], serie.loc[annees].values, color=couleur, width=0.6)
    for x, valeur in enumerate(serie.loc[annees].values):
        ax.text(x, valeur, f"{valeur:,.0f}".replace(",", " "), ha="center", va="bottom", fontweight="bold", color=viz.FONCE)
    evolution = serie.loc[annees[-1]] / serie.loc[annees[0]] - 1
    ax.set_title(f"{libelle}\n({evolution:+.0%} de {annees[0]} à {annees[-1]})".replace("%", " %"), fontsize=10, color=viz.GRIS)
    ax.set_ylim(0, serie.max() * 1.18)
    ax.grid(axis="x", visible=False)
fig.suptitle("L'offre d'un magasin, selon la mesure choisie", x=0.01, y=0.99, ha="left", fontsize=14, fontweight="bold", color=viz.FONCE)
fig.text(0.01, 0.905, "Moyenne par magasin · magasins ouverts toute l'année", ha="left", color=viz.GRIS)
fig.tight_layout(rect=(0, 0, 1, 0.9))
print("Figure enregistrée :", viz.enregistrer(fig, "05c_mesures_de_l_offre"))
plt.show()

**Lecture (à ce stade, on décrit ; l'explication est l'objet du notebook d'analyse de l'offre)**

- **Trois mesures de l'offre d'un magasin, trois baisses très différentes** entre 2019 et 2021 : −73 % pour les produits relevés en stock dans le mois (531, 292, 144), −66 % pour les produits vendus dans le mois (320, 153, 110), −35 % pour les produits vendus dans l'année (1 508, 1 245, 979). La première mesure est écartée (section 3). Les deux autres viennent des ventes, mais ne disent pas la même chose.
- **Les ventes sont enregistrées en lignes moins nombreuses et plus grosses.** Un magasin passe de 359 lignes de vente par mois à 114, mais chaque ligne porte 9,5 unités en 2019, 12,5 en 2020 et 17,1 en 2021. Le nombre de produits vendus dans le mois suit le nombre de lignes (une ligne par produit, ou presque). Une partie de sa baisse vient donc de la façon dont les ventes sont regroupées en lignes, et non du nombre de produits proposés. **Le compte mensuel surestime la réduction de l'offre.**
- **Sur l'année, un magasin vend quand même un tiers de produits en moins** (−35 %). C'est la mesure la plus prudente, et elle reste forte.
- **Le catalogue, lui, s'élargit** : les magasins vendent ensemble 1 729 produits différents en 2019, 2 142 en 2020 et 2 384 en 2021. Ce qui recule, c'est la **diffusion** : un produit est vendu dans 44 magasins en moyenne en 2019, 30 en 2020 et 22 en 2021. Chaque magasin ne vend plus qu'une partie d'un catalogue plus large.
- **Volume et valeur ne bougent pas ensemble.** De 2019 à 2020, un magasin perd 41 % de ses unités et 44 % de son chiffre d'affaires. De 2020 à 2021, il garde ses unités (−3 %) mais perd encore 17 % de son chiffre d'affaires : il vend autant de pièces, moins chères.
- **Les nouveautés** : 552 produits mis en vente en 2019, 429 en 2020, 272 en 2021 (la dernière mise en vente date du 21 juillet 2021). Un produit lancé pendant la période est vendu dans les jours qui suivent sa mise en vente. La date du catalogue est fiable : 23 produits seulement sont vendus avant elle.

Ces constats ne tranchent pas entre l'offre et la demande : ils disent quelles mesures sont utilisables. Le notebook d'analyse de l'offre devra raisonner sur les produits vendus **dans l'année** et sur la **diffusion** des produits entre magasins, et séparer l'effet volume de l'effet prix.

# Partie B · Les objectifs de vente

## 5. Structure et rapprochement avec nos ventes

La table donne, pour chaque boutique, produit, canal et mois, une quantité, un montant de ventes et une marge, dans trois scénarios : 1 = réel, 2 = budget, 3 = prévision (les mêmes scénarios que dans le plan stratégique).

In [ ]:
display(objectifs.head(3))
SCENARIOS = {1: "Réel", 2: "Budget", 3: "Prévision"}
objectifs["scenario"] = objectifs.id_scenario.map(SCENARIOS)

CLE_OBJECTIFS = ["date", "id_boutique", "id_produit", "id_canal", "id_scenario"]
noter("Objectifs", "Lignes en double sur la clé (mois, boutique, produit, canal, scénario)", objectifs.duplicated(CLE_OBJECTIFS).sum(),
      "À corriger", "Dédoublonner", total=len(objectifs))
noter("Objectifs", "Dates qui ne sont pas un premier jour du mois", (objectifs.date.dt.day != 1).sum(), "À corriger", "Ramener au mois")
noter("Objectifs", "Scénarios inconnus", objectifs.scenario.isna().sum(), "À corriger", "Compléter la table des scénarios")
for col in ["quantite_objectif", "ventes_objectif"]:
    noter("Objectifs", f"{col} négatif ou nul", (objectifs[col] <= 0).sum(), "À décider", "Vérifier", total=len(objectifs))
noter("Objectifs", "marge_objectif négative", (objectifs.marge_objectif < 0).sum(), "Information",
      "Marge après retours : quelques cellules où les retours dépassent la marge", total=len(objectifs))

# Une colonne par scénario
cellule = ["date", "id_boutique", "id_produit", "id_canal"]
large = objectifs.pivot_table(index=cellule, columns="scenario", values=["quantite_objectif", "ventes_objectif", "marge_objectif"])
noter("Objectifs", "Cellules (mois, boutique, produit, canal) sans les trois scénarios", large["ventes_objectif"].isna().any(axis=1).sum(),
      "À décider", "Comparer seulement les cellules complètes", total=len(large))

display(objectifs.groupby(["annee", "scenario"]).agg(lignes=("date", "size"), quantite=("quantite_objectif", "sum"),
                                                     ventes=("ventes_objectif", "sum"), marge=("marge_objectif", "sum"))
        .unstack("scenario").style.format("{:,.0f}"))

# ---------------------------------------------------------------------------
# Rapprochement du scénario réel avec nos ventes, cellule par cellule
# ---------------------------------------------------------------------------
ventes_cellule = (ventes.groupby(["mois_debut", "id_boutique", "id_produit", "id_canal"])
                  .agg(quantite_vente=("quantite_vente", "sum"), ca_brut=("ca_brut", "sum"), cout_total=("cout_total", "sum"),
                       montant_retour=("montant_retour", "sum"), marge_nette=("marge_nette", "sum"))
                  .reset_index().rename(columns={"mois_debut": "date"}))
reel = pd.DataFrame({"quantite_reel": large[("quantite_objectif", "Réel")], "ventes_reel": large[("ventes_objectif", "Réel")],
                     "marge_reel": large[("marge_objectif", "Réel")]}).reset_index()
rapprochement = ventes_cellule.merge(reel, on=cellule, how="outer", indicator=True)
print("Cellules présentes des deux côtés :", espaces((rapprochement["_merge"] == "both").sum()),
      "· seulement dans les ventes :", int((rapprochement["_merge"] == "left_only").sum()),
      "· seulement dans les objectifs :", int((rapprochement["_merge"] == "right_only").sum()))
noter("Objectifs", "Cellules où les ventes du scénario réel diffèrent de notre chiffre d'affaires",
      ((rapprochement.ca_brut - rapprochement.ventes_reel).abs() > TOLERANCE).sum() + (rapprochement["_merge"] != "both").sum(),
      "À décider", "Chercher l'origine de l'écart", total=len(rapprochement))
noter("Objectifs", "Cellules où la quantité du scénario réel diffère de la quantité vendue",
      (rapprochement.quantite_vente != rapprochement.quantite_reel).sum(), "À décider", "Chercher l'origine de l'écart", total=len(rapprochement))

# À quoi correspond la marge du scénario réel ? On teste trois définitions.
definitions = {
    "ventes − coût total de la source": rapprochement.ca_brut - rapprochement.cout_total,
    "ventes − coût total de la source − montant des retours": rapprochement.ca_brut - rapprochement.cout_total - rapprochement.montant_retour,
    "notre marge nette (05_variables)": rapprochement.marge_nette,
}
display(pd.DataFrame({"total": {nom: serie.sum() for nom, serie in definitions.items()} | {"marge du scénario réel": rapprochement.marge_reel.sum()},
                      "cellules en écart": {nom: int(((serie - rapprochement.marge_reel).abs() > TOLERANCE).sum()) for nom, serie in definitions.items()}})
        .style.format({"total": "{:,.0f}", "cellules en écart": "{:,.0f}"}, na_rep=""))

**Lecture**

- La table est propre : aucun vide, aucun doublon, les trois scénarios présents dans chaque cellule.
- **Le scénario réel est notre table des ventes**, regroupée par mois : même chiffre d'affaires et même quantité dans les 483 284 cellules, au cent près. Il n'apporte donc aucune information nouvelle, mais il confirme une troisième fois nos totaux (après le plan stratégique).
- **La marge du scénario réel** vaut les ventes moins le coût total de la source moins le montant des retours. Elle hérite du coût total calculé de trois façons (`03_diagnostic`, règle R4) : on garde notre marge nette.

## 6. Nature des données : l'objectif est-il un vrai objectif ?

Un objectif de vente est fixé **avant** les ventes. Il devrait donc s'écarter du réel, tantôt au-dessus, tantôt en dessous, différemment selon les boutiques, les produits et les mois. On calcule le rapport budget ÷ réel et prévision ÷ réel de chaque cellule et on regarde sa dispersion.

In [ ]:
rapports = pd.DataFrame({"budget ÷ réel": large[("ventes_objectif", "Budget")] / large[("ventes_objectif", "Réel")],
                         "prévision ÷ réel": large[("ventes_objectif", "Prévision")] / large[("ventes_objectif", "Réel")]})
rapports["mois"] = rapports.index.get_level_values("date")
rapports["annee"] = rapports.mois.dt.year

print("Rapport objectif ÷ réel, cellule par cellule (boutique, produit, canal, mois) :")
display(rapports.groupby("annee")[["budget ÷ réel", "prévision ÷ réel"]].describe(percentiles=[0.05, 0.5, 0.95])
        .loc[:, (slice(None), ["count", "mean", "std", "5%", "50%", "95%"])].style.format("{:,.4f}"))

# Valeur la plus fréquente du rapport, mois par mois, et part des cellules qui la prennent (à 0,001 près)
def valeur_dominante(serie):
    arrondi = serie.round(3)
    mode = arrondi.mode().iloc[0]
    return pd.Series({"coefficient": mode, "part des cellules": (arrondi == mode).mean()})

dominant = pd.concat({col: rapports.groupby("mois")[col].apply(valeur_dominante).unstack() for col in ["budget ÷ réel", "prévision ÷ réel"]}, axis=1)
# On résume par période de coefficient constant
dominant["periode"] = (dominant[("budget ÷ réel", "coefficient")].diff().abs() > 0.0005).cumsum()
periodes = dominant.groupby("periode").apply(lambda g: pd.Series({
    "du": g.index.min().strftime("%Y-%m"), "au": g.index.max().strftime("%Y-%m"),
    "budget ÷ réel": g[("budget ÷ réel", "coefficient")].iloc[0],
    "part des cellules à ce coefficient (budget)": g[("budget ÷ réel", "part des cellules")].mean(),
    "prévision ÷ réel": g[("prévision ÷ réel", "coefficient")].iloc[0],
    "part des cellules à ce coefficient (prévision)": g[("prévision ÷ réel", "part des cellules")].mean()}))
print("Coefficient dominant, par période :")
display(periodes.style.format({"budget ÷ réel": "{:.3f}", "prévision ÷ réel": "{:.3f}",
                               "part des cellules à ce coefficient (budget)": "{:.1%}",
                               "part des cellules à ce coefficient (prévision)": "{:.1%}"}).hide(axis="index"))

# Le rapport dépend-il de la boutique ? Écart au budget par type de boutique et par année
objectifs["type"] = objectifs.id_boutique.map(type_boutique).map(LIBELLES_TYPES)
par_type = objectifs.pivot_table(index=["annee", "type"], columns="scenario", values="ventes_objectif", aggfunc="sum")
par_type["réel ÷ budget"] = par_type["Réel"] / par_type["Budget"]
par_type["réel ÷ prévision"] = par_type["Réel"] / par_type["Prévision"]
print("Ventes par scénario, par type de boutique :")
display(par_type.style.format({"Réel": "{:,.0f}", "Budget": "{:,.0f}", "Prévision": "{:,.0f}", "réel ÷ budget": "{:.3f}", "réel ÷ prévision": "{:.3f}"}))
par_boutique = objectifs.pivot_table(index=["annee", "id_boutique"], columns="scenario", values="ventes_objectif", aggfunc="sum")
print("Réel ÷ budget par boutique : dispersion entre les boutiques, par année")
display((par_boutique["Réel"] / par_boutique["Budget"]).groupby("annee").describe()[["count", "min", "50%", "max", "std"]].style.format("{:.3f}"))

# Ce budget est-il celui du plan stratégique ?
charges = pd.read_parquet(config.CLEAN / "fait_charges.parquet")
budget_plan = charges[charges.famille_compte == "Ventes"].pivot_table(index="annee", columns="scenario", values="montant", aggfunc="sum")
comparaison = pd.DataFrame({"budget des objectifs": objectifs[objectifs.scenario == "Budget"].groupby("annee").ventes_objectif.sum(),
                            "budget du plan stratégique": budget_plan["Budget"], "ventes réelles": budget_plan["Réel"]})
comparaison["objectifs ÷ réel"] = comparaison["budget des objectifs"] / comparaison["ventes réelles"]
comparaison["plan ÷ réel"] = comparaison["budget du plan stratégique"] / comparaison["ventes réelles"]
print("Deux budgets de ventes pour les mêmes boutiques :")
display(comparaison.style.format({"budget des objectifs": "{:,.0f}", "budget du plan stratégique": "{:,.0f}", "ventes réelles": "{:,.0f}",
                                  "objectifs ÷ réel": "{:.3f}", "plan ÷ réel": "{:.3f}"}))

part_coefficient = float((rapports["budget ÷ réel"].round(3) == rapports.groupby("mois")["budget ÷ réel"].transform(lambda s: s.round(3).mode().iloc[0])).mean())
noter("Objectifs", "Cellules dont le budget vaut le réel multiplié par le coefficient du mois", int(part_coefficient * len(rapports)),
      "Information", "Ne pas présenter l'écart aux objectifs comme une mesure de performance", total=len(rapports), zero_est_ok=False)

**Lecture**

- **Le budget est le réel multiplié par un coefficient.** Dans 96,5 % des cellules, le budget d'un produit dans une boutique vaut exactement ses ventes réelles du mois multipliées par le même nombre : 0,976 de janvier 2019 à juin 2020, 0,812 au second semestre 2020, 0,844 en 2021. La prévision suit la même règle (1,076, puis 1,307, puis 0,891).
- **Le coefficient ne dépend ni de la boutique, ni du produit, ni du canal.** Toutes les boutiques « dépassent leur budget » du même pourcentage : +2,4 % en 2019, +11 % en 2020, +18,4 % en 2021, magasins, boutique en ligne et revendeur confondus. Selon cette table, les magasins dépassent leur objectif de 18,7 % l'année où ils passent en déficit.
- **Ce budget n'est pas celui du plan stratégique** : la base contient deux budgets de ventes différents pour les mêmes boutiques (588 M$ contre 672 M$ en 2021), chacun calculé à partir du réel.
- **Conclusion** : l'objectif est calculé après coup à partir des ventes. Un écart aux objectifs ne mesure rien : il redonne le coefficient. **L'analyse des écarts aux objectifs n'est pas retenue**, ni pour le diagnostic, ni pour la partie prescriptive. La table est conservée dans `data/clean` pour mémoire et pour la traçabilité de cette décision.

# Partie C · Les ventes en ligne détaillées

## 7. Structure et qualité

`FactOnlineSales` est la seule table de la base qui contient un **client** et un **numéro de commande**. Elle ne concerne que la boutique en ligne (boutique 306).

In [ ]:
display(en_ligne.head(3))
print("Boutiques :", sorted(en_ligne.id_boutique.unique()), "· clients :", espaces(en_ligne.id_client.nunique()),
      "· commandes :", espaces(en_ligne.numero_commande.nunique()))

noter("Ventes en ligne", "Identifiants de ligne en double", en_ligne.id_vente_en_ligne.duplicated().sum(), "À corriger", "Dédoublonner", total=len(en_ligne))
noter("Ventes en ligne", "Lignes en double sur (commande, numéro de ligne)", en_ligne.duplicated(["numero_commande", "ligne_commande"]).sum(),
      "À corriger", "Dédoublonner", total=len(en_ligne))
noter("Ventes en ligne", "Commandes rattachées à plusieurs clients",
      (en_ligne.groupby("numero_commande").id_client.nunique() > 1).sum(), "Information", "À connaître pour compter les commandes par client",
      total=en_ligne.numero_commande.nunique())
noter("Ventes en ligne", "Produits absents de la dimension produit", len(set(en_ligne.id_produit) - set(dim_produit.id_produit)),
      "À corriger", "Vérifier la dimension")

# Quantités : que vaut une ligne ?
print("\nQuantité vendue par ligne :", en_ligne.quantite_vente.value_counts().to_dict(),
      "· quantité retournée :", en_ligne.quantite_retour.value_counts().to_dict(),
      "· quantité remisée :", en_ligne.quantite_remise.value_counts().to_dict())
display(pd.crosstab(en_ligne.quantite_vente.rename("quantité vendue"), [en_ligne.quantite_retour.rename("quantité retournée"),
                                                                       en_ligne.quantite_remise.rename("quantité remisée")]))
lignes_retour = en_ligne.quantite_vente == 0
noter("Ventes en ligne", "Lignes de quantité vendue nulle (lignes de retour)", lignes_retour.sum(), "Information",
      "Une ligne de retour porte une quantité vendue de 0 et une quantité retournée de 1", total=len(en_ligne), zero_est_ok=False)

# Règles de gestion, sur les lignes de vente (quantité 1)
vente = en_ligne[~lignes_retour]
r1 = ((vente.montant_ventes - (vente.prix_unitaire * vente.quantite_vente - vente.montant_remise)).abs() > TOLERANCE).sum()
noter("Ventes en ligne", "Lignes de vente où montant ≠ prix × quantité − remise (R1)", r1, "À décider", "Chercher l'origine", total=len(vente))
prix_catalogue = en_ligne.id_produit.map(dim_produit.set_index("id_produit").prix_catalogue)
noter("Ventes en ligne", "Lignes dont le prix unitaire diffère du prix catalogue", ((en_ligne.prix_unitaire - prix_catalogue).abs() > TOLERANCE).sum(),
      "Information", "Aucune", total=len(en_ligne))
print("\nLignes de retour (quantité vendue 0) : montant des ventes total =", espaces(en_ligne.loc[lignes_retour, "montant_ventes"].sum(), 2),
      "· montant des retours =", espaces(en_ligne.loc[lignes_retour, "montant_retour"].sum(), 2),
      "· montant des remises =", espaces(en_ligne.loc[lignes_retour, "montant_remise"].sum(), 2))

## 8. Ce détail est-il celui des ventes de la boutique en ligne ?

Notre table des ventes contient déjà la boutique en ligne (`FactSales`, canal *Online*), et ses totaux sont égaux à ceux du plan stratégique au cent près. Si `FactOnlineSales` en était le détail, la somme de ses lignes redonnerait les mêmes totaux, année par année, mois par mois et produit par produit.

In [ ]:
ventes_306 = ventes[ventes.id_boutique.isin(en_ligne.id_boutique.unique())]


def totaux(table):
    """Totaux annuels d'une table de ventes (mêmes noms de colonnes dans les deux tables)."""
    g = table.groupby("annee")
    resultat = g.agg(lignes=("quantite_vente", "size"), unites=("quantite_vente", "sum"), ventes=("montant_ventes", "sum"),
                     remises=("montant_remise", "sum"), retours=("montant_retour", "sum"), unites_retournees=("quantite_retour", "sum"),
                     produits=("id_produit", "nunique"), promotions=("id_promotion", "nunique"))
    resultat["avant_remise"] = g.apply(lambda d: (d.prix_unitaire * d.quantite_vente).sum(), include_groups=False)
    resultat["taux de remise effectif"] = resultat.remises / resultat.avant_remise
    resultat["taux de retour (unités)"] = resultat.unites_retournees / resultat.unites
    return resultat.drop(columns="avant_remise")


detail, reference = totaux(en_ligne), totaux(ventes_306)
comparaison = pd.concat({"détail (FactOnlineSales)": detail, "ventes (FactSales, boutique en ligne)": reference,
                         "détail ÷ ventes": detail / reference}, axis=0)
display(comparaison.style.format({"lignes": "{:,.2f}", "unites": "{:,.2f}", "ventes": "{:,.2f}", "remises": "{:,.2f}", "retours": "{:,.2f}",
                                  "unites_retournees": "{:,.2f}", "produits": "{:,.2f}", "promotions": "{:,.2f}",
                                  "taux de remise effectif": "{:.3f}", "taux de retour (unités)": "{:.4f}"}))
noter("Ventes en ligne", "Années où le total du détail diffère des ventes de la boutique en ligne",
      ((detail.ventes - reference.ventes).abs() > 1).sum(), "À décider", "Ne pas additionner ni substituer les deux tables", total=len(detail))

# Mois par mois
par_mois = pd.DataFrame({"détail": en_ligne.groupby(en_ligne.date.dt.to_period("M")).montant_ventes.sum(),
                         "ventes": ventes_306.groupby(ventes_306.date.dt.to_period("M")).montant_ventes.sum()})
par_mois["détail ÷ ventes"] = par_mois["détail"] / par_mois["ventes"]
print("Rapport détail ÷ ventes, mois par mois :")
display(par_mois["détail ÷ ventes"].to_frame().assign(annee=lambda d: d.index.year, mois=lambda d: d.index.month)
        .pivot(index="mois", columns="annee", values="détail ÷ ventes").style.format("{:.2f}"))

# Produit par produit : les deux tables classent-elles les produits de la même façon ?
p_detail = en_ligne.groupby(["annee", "id_produit"]).agg(unites=("quantite_vente", "sum"), ventes=("montant_ventes", "sum"))
p_ventes = ventes_306.groupby(["annee", "id_produit"]).agg(unites=("quantite_vente", "sum"), ventes=("montant_ventes", "sum"))
par_produit = p_detail.join(p_ventes, how="outer", lsuffix="_detail", rsuffix="_ventes")
accord = par_produit.groupby("annee").apply(lambda g: pd.Series({
    "produits des deux côtés": int(g.dropna().shape[0]),
    "produits d'un seul côté": int(g.isna().any(axis=1).sum()),
    "corrélation des unités": g.unites_detail.corr(g.unites_ventes),
    "corrélation des montants": g.ventes_detail.corr(g.ventes_ventes),
    "produits à moins de 10 % d'écart (unités)": ((g.unites_detail / g.unites_ventes - 1).abs() < 0.10).mean()}), include_groups=False)
print("Accord produit par produit :")
display(accord.style.format({"produits des deux côtés": "{:,.0f}", "produits d'un seul côté": "{:,.0f}", "corrélation des unités": "{:.3f}",
                             "corrélation des montants": "{:.3f}", "produits à moins de 10 % d'écart (unités)": "{:.1%}"}))

**Lecture**

- **Le détail n'est pas celui de nos ventes.** Ses totaux diffèrent chaque année de ceux de la boutique en ligne : +31 % en 2019, −8 % en 2020, −9 % en 2021. L'écart change d'un mois à l'autre (de 0,82 à 1,94).
- **En 2020 et 2021, les deux tables se ressemblent** : mêmes produits, classés de la même façon (corrélation de 0,99), mais le détail est plus bas de 1 % à 17 % selon le mois. **En 2019, elles ne se ressemblent pas** : le détail est plus haut, et les montants par produit ne sont presque pas liés (corrélation de 0,19).
- **Les remises ne suivent pas la même règle.** Dans le détail, 73 % des unités sont remisées et le taux de remise effectif est de 9,4 %. Dans nos ventes, il est de 2 %. Une analyse des remises donnerait donc des résultats différents selon la table.
- **Une ligne est une unité** : la quantité vaut toujours 1, sauf sur les 33 051 lignes de retour, qui portent une quantité vendue de 0.
- **Conséquence** : on ne remplace pas les ventes de la boutique en ligne par ce détail, et on n'additionne jamais les deux. Notre table des ventes reste la référence pour tous les montants, parce qu'elle seule recoupe le plan stratégique et les objectifs au cent près.

## 9. Les clients et les commandes

C'est ce que cette table apporte et qu'aucune autre ne contient. On regarde qui achète, et si les commandes ressemblent à celles d'un vrai site marchand.

In [ ]:
clients = pd.read_csv(config.DIM / "DimCustomer.csv", encoding="utf-8-sig")
type_client = clients.set_index("CustomerKey").CustomerType.map({"Person": "Particulier", "Company": "Entreprise"})
en_ligne["type_client"] = en_ligne.id_client.map(type_client)
noter("Ventes en ligne", "Lignes dont le client est absent de la dimension client", en_ligne.type_client.isna().sum(),
      "À corriger", "Vérifier la dimension", total=len(en_ligne))

par_client_type = en_ligne.groupby(["annee", "type_client"]).agg(clients=("id_client", "nunique"), commandes=("numero_commande", "nunique"),
                                                                 unites=("quantite_vente", "sum"), ventes=("montant_ventes", "sum"))
par_client_type["part des ventes"] = par_client_type.ventes / par_client_type.groupby("annee").ventes.transform("sum")
par_client_type["ventes par client"] = par_client_type.ventes / par_client_type.clients
par_client_type["unités par commande"] = par_client_type.unites / par_client_type.commandes
display(par_client_type.style.format({"clients": "{:,.0f}", "commandes": "{:,.0f}", "unites": "{:,.0f}", "ventes": "{:,.0f}",
                                      "part des ventes": "{:.1%}", "ventes par client": "{:,.0f}", "unités par commande": "{:,.1f}"}))

# Fidélité : un client acheteur une année l'est-il les autres années ?
annees_par_client = en_ligne.groupby("id_client").annee.nunique()
print("Clients selon le nombre d'années avec au moins un achat :", annees_par_client.value_counts().sort_index().to_dict())
commandes_par_client = en_ligne.groupby(["annee", "type_client", "id_client"]).numero_commande.nunique()
print("Commandes par client et par an :")
display(commandes_par_client.groupby(["annee", "type_client"]).describe()[["count", "min", "25%", "50%", "75%", "max"]].style.format("{:,.0f}"))

# Taille des commandes
commandes = en_ligne.groupby(["annee", "type_client", "numero_commande"]).agg(lignes=("ligne_commande", "size"), montant=("montant_ventes", "sum"))
print("Taille des commandes (nombre de lignes, donc d'unités) :")
display(commandes.groupby(["annee", "type_client"]).lignes.describe(percentiles=[0.5, 0.99])[["count", "50%", "99%", "max"]].style.format("{:,.0f}"))
noter("Ventes en ligne", "Clients qui n'achètent pas les trois années (0 = aucun client perdu ni gagné)",
      (annees_par_client < en_ligne.annee.nunique()).sum(), "Information",
      "Ne pas étudier la fidélité ni l'attrition : tous les clients achètent chaque année", total=len(annees_par_client), zero_est_ok=False)

**Lecture**

- **5 548 clients : 5 505 particuliers et 43 entreprises.** Les entreprises font la moitié des ventes en 2019, puis plus de 90 % en 2020 et 2021, avec des commandes de plusieurs centaines d'unités.
- **Les ventes aux particuliers s'effondrent en 2020** : de 164,8 M$ à 19,3 M$ (−88 %), avec le même nombre de clients. Un particulier dépense 29 900 $ en 2019 et 3 500 $ en 2020. Ce mouvement n'apparaît pas dans nos ventes, où la boutique en ligne progresse de 9 % en 2020 : il confirme que les deux tables ne décrivent pas la même activité en 2019.
- **Les comportements ne sont pas ceux de vrais clients.** Tous les clients, ou presque, achètent les trois années (5 519 sur 5 548) : il n'y a ni client perdu, ni client gagné. Un particulier passe entre 19 et 76 commandes par an, presque toutes d'une seule unité.
- **Conséquence** : la table permet de décrire qui achète en ligne (particuliers ou entreprises), à titre de contexte. Elle ne permet pas d'étudier la fidélité, l'attrition ou le panier moyen, et elle ne se rattache pas à nos montants.

## 10. Synthèse : ce que chaque table permet

In [ ]:
registre_df = pd.DataFrame(registre)
ordre = ["À décider", "À corriger", "Information", "OK"]
print(registre_df.groupby("table").gravite.value_counts().unstack(fill_value=0).reindex(columns=ordre, fill_value=0).to_string())
pd.set_option("display.max_colwidth", 140)
display(registre_df[registre_df.gravite != "OK"].style.format({"nb": "{:,.0f}", "part": "{:.2%}"}, na_rep="").hide(axis="index"))

### Décisions proposées à l'équipe

| Table | Constat | Décision | Usage dans la suite |
|---|---|---|---|
| Stock | Table propre, cohérente avec les dates d'ouverture | Aucune correction | |
| Stock | Relevé partiel, de moins en moins dense : en 2021, 27 % des couples (magasin, produit) vendus n'ont aucun relevé de l'année | Ne jamais déduire l'absence d'un produit de l'absence de relevé ; ne pas compter les produits en stock par semaine | Niveau de stock d'un produit quand il est relevé ; produits relevés sans vente dans l'année |
| Stock | 210 relevés à zéro sur 1,5 million | Pas d'analyse des ruptures | |
| Ventes (mesure de l'offre) | Le nombre de produits vendus par mois suit le nombre de lignes, qui baisse parce que les lignes grossissent | Mesurer l'offre d'un magasin sur l'année, et par la diffusion des produits entre magasins | `assortiment_boutique_mois`, `produit_annee` |
| Objectifs | Scénario réel égal à nos ventes, cellule par cellule | Troisième recoupement de nos totaux | Contrôle seulement |
| Objectifs | Budget et prévision égaux au réel multiplié par un coefficient commun à toutes les boutiques | **Analyse des écarts aux objectifs abandonnée** | Aucun |
| Ventes en ligne détaillées | Totaux différents de nos ventes chaque année, sans lien en 2019 | Ne pas substituer ni additionner ; nos ventes restent la référence | Aucun montant |
| Ventes en ligne détaillées | Règle de remise différente (9,4 % contre 2 %) | Ne pas l'utiliser pour Q2, Q4 et Q5 | |
| Ventes en ligne détaillées | Tous les clients achètent chaque année ; commandes d'une unité | Pas d'analyse de fidélité, d'attrition ni de panier | Contexte : part des entreprises dans les ventes en ligne |

**Ce que ces trois tables changent pour le projet.** Elles n'apportent pas la preuve directe qu'on espérait (un stock qui montrerait les produits absents, des objectifs qui montreraient les magasins en retard). Elles apportent autre chose : la certitude que nos ventes sont la bonne référence, recoupée par trois sources, et une mise en garde précise sur la façon de mesurer l'offre.

## 11. Enregistrement

In [ ]:
config.CLEAN.mkdir(parents=True, exist_ok=True)
config.AGREGATS.mkdir(parents=True, exist_ok=True)

# Colonnes de travail retirées des tables de faits avant l'écriture
fait_stock = stock.drop(columns=["mois_debut", "type"])
fait_objectifs = objectifs.drop(columns=["type"])
fait_en_ligne = en_ligne.copy()

# Contrôles avant écriture : aucune ligne ajoutée ni supprimée, aucun montant modifié
controles = {
    "Stock : même nombre de lignes qu'à la lecture": len(fait_stock) == len(pd.read_parquet(chemins["stock"], columns=["date"])),
    "Objectifs : même nombre de lignes qu'à la lecture": len(fait_objectifs) == len(pd.read_parquet(chemins["objectifs"], columns=["date"])),
    "Ventes en ligne : même nombre de lignes qu'à la lecture": len(fait_en_ligne) == len(pd.read_parquet(chemins["ventes_en_ligne"], columns=["date"])),
    "Assortiment : une ligne par boutique et par mois": not assortiment.duplicated(["id_boutique", "mois_debut"]).any(),
    "Assortiment : chiffre d'affaires égal à celui de la table des ventes": abs(assortiment.ca_brut.sum() - ventes.ca_brut.sum()) < 1,
    "Assortiment : nombre de relevés égal à celui de la table de stock": int(assortiment.releves.sum()) == len(stock),
    "Produit par année : une ligne par produit, type de boutique et année": not produit_annee.duplicated(["id_produit", "type", "annee"]).any(),
    "Produit par année : chiffre d'affaires égal à celui de la table des ventes": abs(produit_annee.ca_brut.sum() - ventes.ca_brut.sum()) < 1,
    "Objectifs : ventes du scénario réel égales à notre chiffre d'affaires": abs(
        fait_objectifs.loc[fait_objectifs.scenario == "Réel", "ventes_objectif"].sum() - ventes.ca_brut.sum()) < 1,
}
for nom, ok in controles.items():
    print(f"[{'réussi' if ok else 'ÉCHOUÉ'}] {nom}")
assert all(controles.values()), "Validation échouée : rien n'est écrit."

sorties = {
    config.CLEAN / "fait_stock.parquet": fait_stock,
    config.CLEAN / "fait_objectifs.parquet": fait_objectifs,
    config.CLEAN / "fait_ventes_en_ligne.parquet": fait_en_ligne,
    config.AGREGATS / "assortiment_boutique_mois.parquet": assortiment,
    config.AGREGATS / "produit_annee.parquet": produit_annee,
}
print()
for chemin, table in sorties.items():
    table.to_parquet(chemin, index=False)
    print(f"{chemin.relative_to(config.ROOT)}  ({espaces(len(table))} lignes × {table.shape[1]} colonnes)")

# Rapport de diagnostic (pièce du livrable L3), lisible sans ouvrir le notebook
chemin_rapport = config.L3 / "diagnostic_autres_tables.xlsx"
with pd.ExcelWriter(chemin_rapport) as classeur:
    registre_df.to_excel(classeur, sheet_name="Registre", index=False)
    par_annee.to_excel(classeur, sheet_name="Stock_par_annee")
    densite.to_excel(classeur, sheet_name="Stock_densite_releves")
    couverture_mois.to_excel(classeur, sheet_name="Stock_vs_ventes_mois")
    couverture_an.to_excel(classeur, sheet_name="Stock_vs_ventes_annee")
    resume.to_excel(classeur, sheet_name="Offre_par_magasin")
    periodes.to_excel(classeur, sheet_name="Objectifs_coefficients", index=False)
    comparaison.to_excel(classeur, sheet_name="En_ligne_detail_vs_ventes")
    accord.to_excel(classeur, sheet_name="En_ligne_par_produit")
    par_client_type.to_excel(classeur, sheet_name="En_ligne_clients")
print("\nRapport enregistré :", chemin_rapport.relative_to(config.ROOT))

**Suite** : le notebook d'analyse de l'offre (`06b`) lit `assortiment_boutique_mois` et `produit_annee` pour tester la thèse : les magasins vendent moins parce que leur offre s'est réduite. Les objectifs et le détail des ventes en ligne ne sont pas utilisés pour les questions Q1 à Q5, pour les raisons données en section 10.